# Barkeep Protocol — Act 1.2 Training Notebook
All diagnostics saved via `run_all_diagnostics`.

## Imports

In [1]:
from dataset import make_dataset
from model import Transformer
from train import train_model, TrainConfig
from analysis import run_all_diagnostics, load_runs, plot_run_comparison

import torch

## Config

In [2]:
# ── device ──
if torch.xpu.is_available():
    DEVICE = "xpu"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

Using device: xpu


In [5]:
# ── reproducibility ──
SEED = 42
generator_global = torch.Generator(device=DEVICE).manual_seed(SEED)

In [6]:
# ── dataset ──
CONTEXT_SIZE = 10
DATA_FILE    = "../Names.txt"
ds = make_dataset(DATA_FILE, DEVICE, context_size=CONTEXT_SIZE, seed=SEED)

trn       = ds["trn"]
dev       = ds["dev"]
test      = ds["test"]

VOCAB_SIZE = ds["vocab_size"]

print(f"Vocab size: {VOCAB_SIZE}")
print("Train:", trn.stats()['examples'])
print("Dev:  ", dev.stats()['examples'])

Vocab size: 28
Train: 25166
Dev:   3146


In [7]:
# ── architecture hyperparameters ──
EMBEDDING_DIM  = 30
NUM_HEADS      = 5
DIM_QK         = 10   # d_k per head
DIM_V          = 20   # d_v per head
FFN_DIM        = 100
NUM_BLOCKS     = 3
DROPOUT        = 0.2
MAX_NORM       = 1
BETA_1         = 0.9
BETA_2         = 0.99
EPSILON        = 1e-8
DECAY_RATE     = 0.1

# ── training hyperparameters ──
BATCH_SIZE     = 4096
STARTING_LR    = 3e-3
ENDING_LR      = 3e-4
WARMUP_STEPS   = 1000
NUM_ITRNS      = 30000

config = TrainConfig(
    log_path     = "runs.jsonl",
    lr_start     = STARTING_LR,
    lr_end       = ENDING_LR,
    iterations   = NUM_ITRNS,
    log_interval = 3000,
    batch_size   = BATCH_SIZE,
    max_norm     = MAX_NORM,
    warmup_steps = WARMUP_STEPS,
    beta_1       = BETA_1,
    beta_2       = BETA_2,
    epsilon      = EPSILON,
    decay_coeff  = DECAY_RATE,
)

# ── diagnostic batch (shared across all models) ──
DIAG_X = trn.inputs[:512]
DIAG_Y = trn.targets[:512]

---
## Transformer

In [6]:
optimized_transformer = Transformer(
    CONTEXT_SIZE, VOCAB_SIZE, EMBEDDING_DIM,
    dim_qk=DIM_QK,
    dim_v=DIM_V,
    num_heads=NUM_HEADS,
    ffn_dim=FFN_DIM,
    n_blocks=NUM_BLOCKS,
    device=DEVICE, generator=generator_global, p=DROPOUT,
)
print("Optimized Transformer\n", optimized_transformer.config_dict())
print(f"Parameters:     {sum(p.nelement() for p in optimized_transformer.parameters()):,}")

Optimized Transformer
 {'vocab_dim': 28, 'embedding_dim': 30, 'context_size': 10, 'dim_qk': 10, 'dim_v': 20, 'num_heads': 5, 'ffn_dim': 100, 'n_blocks': 3}
Parameters:     46,950


In [7]:
optimized_transformer_results = train_model(SEED, optimized_transformer, trn, dev, config, "Optimized Transformer", DEVICE)
torch.save(optimized_transformer_results, "Training results/optimized_transformer_results.pt")
torch.save(optimized_transformer, "Model Instances/optimized_transformer.pt")

  step   3,000 | dev 2.0327 vs train 2.0660| lr 0.0030
  step   6,000 | dev 2.0145 vs train 2.0446| lr 0.0028
  step   9,000 | dev 2.0159 vs train 2.0256| lr 0.0025
  step  12,000 | dev 2.0154 vs train 2.0296| lr 0.0021
  step  15,000 | dev 2.0131 vs train 2.0107| lr 0.0017
  step  18,000 | dev 2.0143 vs train 2.0021| lr 0.0013
  step  21,000 | dev 2.0001 vs train 2.0106| lr 0.0009
  step  24,000 | dev 2.0081 vs train 2.0059| lr 0.0006
  step  27,000 | dev 2.0029 vs train 2.0084| lr 0.0004
  step  30,000 | dev 2.0080 vs train 2.0068| lr 0.0003


In [9]:
run_all_diagnostics(optimized_transformer, optimized_transformer_results, DIAG_X, DIAG_Y, "Optimized Transformer")
runs = load_runs("runs.jsonl")
plot_run_comparison(runs, metric="dev_loss")
plot_run_comparison(runs, metric="train_loss")

── Optimized Transformer ──
  saved to D:\Bar-Eden\Act 1\Act 1.2 Transformer\Plots\Optimized Transformer
